In [1]:
import os
import torch
from google.colab import drive

# ==========================================
# 1. Verify GPU Allocation & CUDA Support
# ==========================================
print("Checking Hardware Environment...")
if not torch.cuda.is_available():
    raise SystemError(
        "ERROR: No GPU detected! Please go to 'Runtime -> Change runtime type' "
        "and select 'T4 GPU' as your Hardware Accelerator."
    )

gpu_name = torch.cuda.get_device_name(0)
total_vram = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
compute_capability = torch.cuda.get_device_capability(0)

print(f"✅ GPU Detected: {gpu_name}")
print(f"✅ Total VRAM: {total_vram} GB")
print(f"✅ CUDA Compute Capability: {compute_capability[0]}.{compute_capability[1]}")

# ==========================================
# 2. Mount Google Drive for Persistent Storage
# ==========================================
print("\nMounting Google Drive...")
drive.mount('/content/drive')

# ==========================================
# 3. Create Structured Project Directories
# ==========================================
PROJECT_DIR = "/content/drive/MyDrive/Nutritionist_LLM_Phase1"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
MODEL_EXPORT_DIR = os.path.join(PROJECT_DIR, "exported_models")

for directory in [PROJECT_DIR, DATA_DIR, CHECKPOINT_DIR, MODEL_EXPORT_DIR]:
    os.makedirs(directory, exist_ok=True)

print(f"\n✅ Project workspace initialized successfully at: {PROJECT_DIR}")
print(f"   ├── Data Folder: {DATA_DIR}")
print(f"   ├── Checkpoints Folder: {CHECKPOINT_DIR}")
print(f"   └── Exported Models Folder: {MODEL_EXPORT_DIR}")

Checking Hardware Environment...
✅ GPU Detected: Tesla T4
✅ Total VRAM: 14.56 GB
✅ CUDA Compute Capability: 7.5

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Project workspace initialized successfully at: /content/drive/MyDrive/Nutritionist_LLM_Phase1
   ├── Data Folder: /content/drive/MyDrive/Nutritionist_LLM_Phase1/data
   ├── Checkpoints Folder: /content/drive/MyDrive/Nutritionist_LLM_Phase1/checkpoints
   └── Exported Models Folder: /content/drive/MyDrive/Nutritionist_LLM_Phase1/exported_models


In [2]:
# ==========================================
# 1. Remove Conflicting xformers & Install Unsloth
# ==========================================
print("⏳ Configuring Unsloth and QLoRA packages for PyTorch 2.11 + CUDA 13.0...")

# Remove xformers so it doesn't clash with torch 2.11.0+cu130 (PyTorch native SDPA will be used)
!pip uninstall -y -q xformers

# Install Unsloth and core QLoRA dependencies cleanly
!pip install -q --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

# ==========================================
# 2. Import Unsloth FIRST (Crucial for Zero Warnings!)
# ==========================================
import unsloth  # MUST be imported before torch, trl, transformers, or peft!
import os
import torch
import trl
import peft
import bitsandbytes
import datasets

print("\n✅ Cell 2 Complete! Zero xformers or import-order errors:")
print(f"   ├── PyTorch Version      : {torch.__version__} (Using Native PyTorch SDPA)")
print(f"   ├── TRL Version          : {trl.__version__}")
print(f"   ├── PEFT Version         : {peft.__version__}")
print(f"   ├── BitsAndBytes Version : {bitsandbytes.__version__}")
print(f"   └── Datasets Version     : {datasets.__version__}")

⏳ Configuring Unsloth and QLoRA packages for PyTorch 2.11 + CUDA 13.0...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

✅ Cell 2 Complete! Zero xformers or import-order errors:
   ├── PyTorch Version      : 2.11.0+cu130 (Using Native PyTorch SDPA)
   ├── TRL Version          : 1.15.0
   ├── PEFT Version         : 0.21.2
   ├── BitsAndBytes Version : 0.50.2
   └── Datasets Version     : 4.8.5


In [3]:
import os
import re
import json
import pandas as pd
from datasets import load_dataset, Dataset

# ==========================================
# 1. Load Raw Medical Dataset from Hugging Face
# ==========================================
print("⏳ Downloading medical dialogue dataset from Hugging Face Hub...")
raw_hf_dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")

# Convert to Pandas DataFrame for fast vectorized regex filtering
df = raw_hf_dataset.to_pandas()
print(f"✅ Total raw medical dialogues loaded: {len(df):,}")

# ==========================================
# 2. Filter for Clinical Nutrition & Dietetics
# ==========================================
df = df.dropna(subset=["input", "output"]).copy()
df = df.rename(columns={"input": "patient_query", "output": "clinical_response"})

# Regex pattern targeting nutrition, macros, micronutrients, and metabolic conditions
nutrition_keywords = (
    r"\b(diet|nutrition|nutritionist|dietitian|calorie|calories|protein|carbohydrate|carbs|"
    r"keto|ketogenic|fasting|vitamin|mineral|iron|calcium|omega|fatty acid|fiber|cholesterol|"
    r"glycemic|insulin|diabetes|pcos|thyroid|celiac|gluten|lactose|ibs|fodmap|obesity|"
    r"weight loss|underweight|bmi|bmr|metabolism|hydration|electrolyte|supplement|creatine|"
    r"whey|vegan|vegetarian|anemia|hypertension|sodium|potassium|renal diet)\b"
)

# Keep rows where BOTH patient query and doctor response explicitly discuss nutrition/diet
mask_query = df["patient_query"].str.contains(nutrition_keywords, case=False, regex=True, na=False)
mask_resp = df["clinical_response"].str.contains(nutrition_keywords, case=False, regex=True, na=False)
df_nutrition = df[mask_query & mask_resp].copy()

print(f"✅ Filtered clinical nutrition consultations: {len(df_nutrition):,}")

# ==========================================
# 3. Clean Scraping Artifacts & Forum Watermarks
# ==========================================
def clean_clinical_text(text: str) -> str:
    """Removes forum watermarks, generic salutations, and irregular whitespace."""
    # Strip website watermarks present in the raw dataset
    text = re.sub(r"Welcome to Chat\s*Doctor[!.,\s]*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Chat\s*Doctor", "our nutrition clinic", text, flags=re.IGNORECASE)
    text = re.sub(r"HealthCareMagic", "clinical guidelines", text, flags=re.IGNORECASE)
    
    # Strip repetitive forum greetings so the bot responds directly and professionally
    text = re.sub(
        r"^(Hi|Hello|Dear\s+Patient|Greetings)[,!.\s]*(Thank you for (posting|asking|your) (query|question)[.!,\s]*)?",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"^I have gone through your (query|question) and understand your concern[.!,\s]*",
        "",
        text,
        flags=re.IGNORECASE,
    )
    return re.sub(r"\s+", " ", text).strip()

df_nutrition["patient_query"] = df_nutrition["patient_query"].apply(clean_clinical_text)
df_nutrition["clinical_response"] = df_nutrition["clinical_response"].apply(clean_clinical_text)

# ==========================================
# 4. Quality Control & Sampling (1,500 Rows)
# ==========================================
df_nutrition = df_nutrition.drop_duplicates(subset=["patient_query"]).copy()
df_nutrition["query_len"] = df_nutrition["patient_query"].str.len()
df_nutrition["resp_len"] = df_nutrition["clinical_response"].str.len()

# Keep detailed clinical answers (150 to 2,500 characters fits easily inside 2048 tokens)
df_clean = df_nutrition[
    (df_nutrition["query_len"].between(30, 1500)) &
    (df_nutrition["resp_len"].between(150, 2500))
].copy()

# Sample 1,500 high-quality consultations for optimal T4 training speed (~15-20 mins)
SAMPLE_SIZE = min(1500, len(df_clean))
df_sampled = df_clean.sample(n=SAMPLE_SIZE, random_state=3407).reset_index(drop=True)

# ==========================================
# 5. Format to ChatML & Create 90/10 Train-Val Split
# ==========================================
SYSTEM_PROMPT = (
    "You are an empathetic, evidence-based clinical nutritionist. "
    "Provide scientifically accurate dietary, macronutrient, and micronutrient guidance. "
    "Never diagnose severe medical conditions (e.g., CKD, Type 1 Diabetes); always refer such patients to a physician."
)

def to_chatml_conversation(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["patient_query"]},
        {"role": "assistant", "content": row["clinical_response"]}
    ]

df_sampled["conversations"] = df_sampled.apply(to_chatml_conversation, axis=1)

# Save permanent JSONL backup to your mounted Google Drive
jsonl_path = os.path.join(DATA_DIR, "nutrition_chatml_cleaned.jsonl")
df_sampled[["conversations"]].to_json(jsonl_path, orient="records", lines=True)

# Convert back to Hugging Face Dataset and split 90% Train (1,350) / 10% Validation (150)
full_hf_dataset = Dataset.from_pandas(df_sampled[["conversations"]])
dataset_splits = full_hf_dataset.train_test_split(test_size=0.1, shuffle=True, seed=3407)

train_dataset = dataset_splits["train"]
eval_dataset = dataset_splits["test"]

print(f"\n✅ Saved permanent JSONL backup to: {jsonl_path}")
print(f"✅ Training Split: {len(train_dataset)} rows | Validation Split: {len(eval_dataset)} rows")
print("\n🔍 Sample Cleaned ChatML Record:\n", json.dumps(train_dataset[0], indent=2))

⏳ Downloading medical dialogue dataset from Hugging Face Hub...
✅ Total raw medical dialogues loaded: 112,165


/tmp/ipykernel_4219/1696528464.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_query = df["patient_query"].str.contains(nutrition_keywords, case=False, regex=True, na=False)
/tmp/ipykernel_4219/1696528464.py:34: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_resp = df["clinical_response"].str.contains(nutrition_keywords, case=False, regex=True, na=False)


✅ Filtered clinical nutrition consultations: 4,836

✅ Saved permanent JSONL backup to: /content/drive/MyDrive/Nutritionist_LLM_Phase1/data/nutrition_chatml_cleaned.jsonl
✅ Training Split: 1350 rows | Validation Split: 150 rows

🔍 Sample Cleaned ChatML Record:
 {
  "conversations": [
    {
      "content": "You are an empathetic, evidence-based clinical nutritionist. Provide scientifically accurate dietary, macronutrient, and micronutrient guidance. Never diagnose severe medical conditions (e.g., CKD, Type 1 Diabetes); always refer such patients to a physician.",
      "role": "system"
    },
    {
      "content": "I had an ultrasound that says that I have mild right hydronephrosis. The urologist wants to order a CT scan but I m hestitant in getting that done because I ve had so many of them already. I ve had 6 CTs in my lifetime with and without contrast. I have thyroid issues and want to avoid contrast. Aren t there other tests that can be done to diagnose me? Cystoscopy? Please advi

In [4]:
from unsloth import FastLanguageModel
import torch

# ==========================================
# 1. Define Model & Quantization Parameters
# ==========================================
max_seq_length = 2048 # Context window size (2048 tokens fits comfortably in T4 VRAM)
dtype = None          # Auto-detects float16 (fp16) for Tesla T4 GPU
load_in_4bit = True   # Enables 4-bit NormalFloat (NF4) quantization via bitsandbytes

print("⏳ Downloading 4-bit Llama 3.1 (8B) base model to T4 GPU...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# ==========================================
# 2. Inject LoRA Adapters (PEFT Configuration)
# ==========================================
print("\n⏳ Injecting trainable LoRA adapters into attention & MLP layers...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA Rank: 16 provides optimal capacity for persona + clinical terminology
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj", # Self-Attention projection matrices
        "gate_proj", "up_proj", "down_proj",    # SwiGLU Feed-Forward (MLP) matrices
    ],
    lora_alpha = 16,                        # Scaling factor (alpha / r = 1.0)
    lora_dropout = 0,                       # Must be 0 to enable Unsloth's fast Triton kernels
    bias = "none",                          # Must be "none" for optimal VRAM efficiency
    use_gradient_checkpointing = "unsloth", # Offloads intermediate activations, saving ~30% VRAM
    random_state = 3407,
    use_rslora = False,
)

# ==========================================
# 3. Audit Trainable Parameters & GPU VRAM
# ==========================================
print("\n✅ Model & LoRA Adapters Initialized Successfully!")
model.print_trainable_parameters()

allocated_vram = round(torch.cuda.memory_allocated(0) / (1024**3), 2)
reserved_vram = round(torch.cuda.memory_reserved(0) / (1024**3), 2)
print(f"📊 Current GPU VRAM Usage -> Allocated: {allocated_vram} GB | Reserved: {reserved_vram} GB / 14.75 GB")

⏳ Downloading 4-bit Llama 3.1 (8B) base model to T4 GPU...
==((====))==  Unsloth 2026.10.11: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-bnb-4bit as a legacy tokenizer.



⏳ Injecting trainable LoRA adapters into attention & MLP layers...


Unsloth 2026.10.11 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).

✅ Model & LoRA Adapters Initialized Successfully!
trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
📊 Current GPU VRAM Usage -> Allocated: 5.48 GB | Reserved: 5.5 GB / 14.75 GB


In [5]:
from unsloth.chat_templates import get_chat_template

# ==========================================
# 1. Bind ChatML Template to the Tokenizer
# ==========================================
print("⏳ Configuring tokenizer with ChatML syntax...")
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Enforces standard <|im_start|> and <|im_end|> delimiters
    mapping = {"role": "role", "content": "content", "user": "user", "assistant": "assistant"}
)

# ==========================================
# 2. Define Formatting Mapping Function
# ==========================================
def formatting_prompts_func(examples):
    """Converts conversation dicts into flat tokenizable strings using ChatML."""
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, 
            tokenize=False,            # Return string first (tokenization happens inside SFTTrainer)
            add_generation_prompt=False # False during training so model learns to output <|im_end|>
        ) 
        for convo in convos
    ]
    return { "text" : texts }

# ==========================================
# 3. Apply Mapping to Train & Eval Datasets
# ==========================================
print("⏳ Applying chat template across training and validation splits...")
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True)

print("\n✅ Tokenizer Setup Complete!")
print(f"   ├── Formatted Training Rows   : {len(train_dataset)}")
print(f"   └── Formatted Validation Rows : {len(eval_dataset)}")

# Print a preview of how Llama 3.1 will actually "see" the training text
print("\n🔍 Preview of Flattened ChatML Training String:\n")
print(train_dataset[0]["text"][:650] + "\n... [TRUNCATED FOR PREVIEW] ...")

⏳ Configuring tokenizer with ChatML syntax...


Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_mjkjtd9w/tokenizer_config.json.


⏳ Applying chat template across training and validation splits...


Map:   0%|          | 0/1350 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]


✅ Tokenizer Setup Complete!
   ├── Formatted Training Rows   : 1350
   └── Formatted Validation Rows : 150

🔍 Preview of Flattened ChatML Training String:

<|im_start|>system
You are an empathetic, evidence-based clinical nutritionist. Provide scientifically accurate dietary, macronutrient, and micronutrient guidance. Never diagnose severe medical conditions (e.g., CKD, Type 1 Diabetes); always refer such patients to a physician.<|im_end|>
<|im_start|>user
I had an ultrasound that says that I have mild right hydronephrosis. The urologist wants to order a CT scan but I m hestitant in getting that done because I ve had so many of them already. I ve had 6 CTs in my lifetime with and without contrast. I have thyroid issues and want to avoid contrast. Aren t there other tests that can be done to diag
... [TRUNCATED FOR PREVIEW] ...


In [6]:
import os
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

# ==========================================
# 1. Define Training Checkpoint Directory
# ==========================================
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ==========================================
# 2. Configure Training Arguments
# ==========================================
training_args = TrainingArguments(
    # Batch Size & Gradient Accumulation (Simulates larger batch size on T4 VRAM)
    per_device_train_batch_size = 2,     # 2 samples per forward pass (fits in ~7GB VRAM)
    gradient_accumulation_steps = 4,     # Accumulate gradients over 4 steps -> Effective Batch Size = 8
    
    # Learning Rate & Schedule
    learning_rate = 2e-4,                # Standard optimal learning rate for QLoRA fine-tuning
    lr_scheduler_type = "linear",        # Linear decay schedule
    warmup_steps = 10,                   # Gentle warmup to stabilize initial weight updates
    
    # Epochs & Steps
    num_train_epochs = 3,                # 3 complete passes over the 1,350 training records
    
    # Hardware Optimization & Precision
    fp16 = not torch.cuda.is_bf16_supported(), # True on Tesla T4 (Compute Capability 7.5)
    bf16 = torch.cuda.is_bf16_supported(),     # False on T4 (reserved for A100/L4 GPUs)
    optim = "adamw_8bit",                      # 8-bit AdamW optimizer saves ~2GB of optimizer VRAM
    weight_decay = 0.01,
    seed = 3407,
    
    # Logging, Evaluation & Saving Strategy
    logging_steps = 10,                        # Print training loss every 10 steps
    eval_strategy = "steps",                   # Evaluate against validation set periodically
    eval_steps = 50,                           # Run validation evaluation every 50 steps
    save_strategy = "steps",                   # Save checkpoints to Google Drive periodically
    save_steps = 50,                           # Save checkpoint folder every 50 steps
    output_dir = CHECKPOINT_DIR,
    report_to = "none",                        # Disable external loggers like wandb
)

# ==========================================
# 3. Initialize the SFT Trainer
# ==========================================
print("⏳ Initializing Supervised Fine-Tuning Trainer...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False, # Keep False to ensure conversation boundaries remain intact
    args = training_args,
)

# ==========================================
# 4. Audit VRAM Before Launching Training
# ==========================================
gpu_stats = torch.cuda.get_device_properties(0)
allocated_vram = round(torch.cuda.memory_allocated(0) / (1024**3), 2)
print(f"📊 Ready to Train! VRAM Allocated: {allocated_vram} GB / {round(gpu_stats.total_memory / (1024**3), 2)} GB")

# ==========================================
# 5. Execute Training Loop
# ==========================================
print("🚀 Starting training run (Estimated time: ~25-35 minutes)...")
trainer_stats = trainer.train()
print("🎉 Training Completed Successfully!")

⏳ Initializing Supervised Fine-Tuning Trainer...


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1350 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/150 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
📊 Ready to Train! VRAM Allocated: 5.48 GB / 14.56 GB
🚀 Starting training run (Estimated time: ~25-35 minutes)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,350 | Num Epochs = 3 | Total steps = 507
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [7]:
import os
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")

# Configure training arguments (matching your previous run)
training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    learning_rate = 2e-4,
    lr_scheduler_type = "linear",
    warmup_steps = 10,
    num_train_epochs = 3,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    optim = "adamw_8bit",
    weight_decay = 0.01,
    seed = 3407,
    logging_steps = 10,
    eval_strategy = "steps",
    eval_steps = 50,
    save_strategy = "steps",
    save_steps = 50,
    output_dir = CHECKPOINT_DIR,
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = training_args,
)

print("🚀 Resuming training from the latest checkpoint on Google Drive...")
# resume_from_checkpoint=True automatically detects the latest checkpoint folder (e.g., checkpoint-300)
trainer.train(resume_from_checkpoint=True)

print("🎉 Training Completed Successfully!")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1350 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/150 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
🚀 Resuming training from the latest checkpoint on Google Drive...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,350 | Num Epochs = 3 | Total steps = 507
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss,Validation Loss
400,1.510035,1.900765
450,1.593974,1.895024
500,1.615517,1.893782
507,1.615517,1.893742


Unsloth: Not an error, but `num_items_in_batch` could not be counted for LlamaForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Nutritionist_LLM_Phase1/checkpoints/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Nutritionist_LLM_Phase1/checkpoints/checkpoint-450/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Nutritionist_LLM_Phase1/checkpoints/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Nutritionist_LLM_Phase1/checkpoints/checkpoint-507/tokenizer_config.json.


🎉 Training Completed Successfully!


In [8]:
import os
import torch
from unsloth import FastLanguageModel
from transformers import TextStreamer

# ==========================================
# 1. Enable Native Fast Inference Mode
# ==========================================
print("⏳ Enabling Unsloth fast inference kernels...")
FastLanguageModel.for_inference(model)

# Define the exact System Prompt used during your training
SYSTEM_PROMPT = (
    "You are an empathetic, evidence-based clinical nutritionist. "
    "Provide scientifically accurate dietary, macronutrient, and micronutrient guidance. "
    "Never diagnose severe medical conditions (e.g., CKD, Type 1 Diabetes); always refer such patients to a physician."
)

test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "I am a vegetarian runner feeling fatigued every afternoon. What key nutrients should I check in my diet?"}
]

# Apply the ChatML template for inference
inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True, # Appends <|im_start|>assistant\n to tell the model it's time to reply
    return_tensors = "pt",
).to("cuda")

print("\n💬 --- LIVE TEST INFERENCE OUTPUT ---")
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    input_ids = inputs, 
    streamer = text_streamer, 
    max_new_tokens = 300,
    temperature = 0.3, # Low temperature ensures grounded, consistent clinical answers
    use_cache = True
)
print("💬 ------------------------------------\n")

# ==========================================
# 2. Export Model to GGUF Format for Local Use
# ==========================================
MODEL_EXPORT_DIR = os.path.join(PROJECT_DIR, "exported_models")
os.makedirs(MODEL_EXPORT_DIR, exist_ok=True)

print(f"⏳ Exporting fine-tuned model to GGUF format (q4_k_m) in {MODEL_EXPORT_DIR}...")
model.save_pretrained_gguf(
    MODEL_EXPORT_DIR, 
    tokenizer, 
    quantization_method = "q4_k_m" # Best balance of high clinical accuracy and low file size (~4.8 GB)
)

print("\n🎉 PHASE 1 COMPLETE!")
print(f"✅ GGUF model successfully saved to your Google Drive folder: {MODEL_EXPORT_DIR}")

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⏳ Enabling Unsloth fast inference kernels...

💬 --- LIVE TEST INFERENCE OUTPUT ---
Thanks for your question on our nutrition clinic. I can understand your concern. Fatigue in the afternoon is common in many diseases. But in your case it seems to be due to nutritional deficiency. You should go for complete blood count, serum ferritin, vitamin B12 and folic acid levels. If any of these levels are low then you should start supplements of that particular vitamin or iron. You should also take multivitamin tablets. Hope I have answered your question, if you have doubt then I will be happy to answer. Thanks for using our nutrition clinic. Wish you a very good health.<|im_end|>
💬 ------------------------------------

⏳ Exporting fine-tuned model to GGUF format (q4_k_m) in /content/drive/MyDrive/Nutritionist_LLM_Phase1/exported_models...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/947 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Nutritionist_LLM_Phase1/exported_models/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


... (12 outputs left out)

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [07:50<00:00, 117.58s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/Nutritionist_LLM_Phase1/exported_models`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11985-mix-c116c32 (app-b11985-mix-c116c32-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...


RuntimeError: Unsloth: GGUF conversion failed: RuntimeError: Unsloth: Failed to convert model to GGUF with command `/usr/bin/python3 /root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py --outfile /content/drive/MyDrive/Nutritionist_LLM_Phase1/exported_models_gguf/.unsloth_gguf_convert_phc1xf4y/Meta-Llama-3.1-8B.F16.gguf --outtype f16 --split-max-size 50G /content/drive/MyDrive/Nutritionist_LLM_Phase1/exported_models`: Command '['/usr/bin/python3', '/root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py', '--outfile', '/content/drive/MyDrive/Nutritionist_LLM_Phase1/exported_models_gguf/.unsloth_gguf_convert_phc1xf4y/Meta-Llama-3.1-8B.F16.gguf', '--outtype', 'f16', '--split-max-size', '50G', '/content/drive/MyDrive/Nutritionist_LLM_Phase1/exported_models']' returned non-zero exit status 1.
--- converter stderr ---
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' i

In [9]:
import os

# Define a local directory on Colab's fast virtual SSD (NOT Google Drive)
LOCAL_SAVE_DIR = "/content/merged_nutritionist_model"

print("⏳ Merging LoRA adapters with base model and saving locally...")
model.save_pretrained_merged(
    LOCAL_SAVE_DIR, 
    tokenizer, 
    save_method = "merged_16bit", # Merges weights cleanly without GGUF conversion errors
)

print(f"\n✅ Model successfully saved locally to: {LOCAL_SAVE_DIR}")

# Optionally copy the resulting folder to your Google Drive for safe keeping
DRIVE_BACKUP_DIR = os.path.join(PROJECT_DIR, "merged_model_drive")
!cp -r {LOCAL_SAVE_DIR} {DRIVE_BACKUP_DIR}
print(f"✅ Backup copy successfully uploaded to Google Drive: {DRIVE_BACKUP_DIR}")

⏳ Merging LoRA adapters with base model and saving locally...


Unsloth: Restored added_tokens_decoder metadata in /content/merged_nutritionist_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


... (8 outputs left out)


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [03:27<00:00, 51.88s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [03:56<00:00, 59.09s/it]


Unsloth: Merge process complete. Saved to `/content/merged_nutritionist_model`

✅ Model successfully saved locally to: /content/merged_nutritionist_model
✅ Backup copy successfully uploaded to Google Drive: /content/drive/MyDrive/Nutritionist_LLM_Phase1/merged_model_drive


In [10]:
import shutil
import os

# Path to the failed GGUF export folder on Google Drive
failed_export_dir = os.path.join(PROJECT_DIR, "exported_models") # or exported_models_gguf

if os.path.exists(failed_export_dir):
    shutil.rmtree(failed_export_dir)
    print(f"🗑️ Successfully deleted obsolete folder: {failed_export_dir}")
else:
    print("📁 Folder already cleaned up.")

🗑️ Successfully deleted obsolete folder: /content/drive/MyDrive/Nutritionist_LLM_Phase1/exported_models
